# Prototipado

No somos desarrolladores. Sabemos construir modelos basados en Python y podemos hacer algo de código pero construir una aplicación que muestre a los usuarios finales que pinta podría tener una aplicación que use nuestro modelo a veces puede ser todo un reto.

No podemos enseñarles simples notebooks con código a nuestros clientes. Hay parte que podemos aterrizar en una presentación pero queremos que no tengan que usar demasiado su imaginación y poner nuestro modelo en términos que ellos puedan entender.

Dada esta necesidad se han creado librerías/frameworks de Python que nos permiten desarrollar interfaces simples y mostrar qué pinta tendría nuestro modelo una vez puesto en marcha.

## Streamlit

Quizás la más popular, cuenta con una base de usuarios muy amplia y permite realizar cosas muy complejas de forma sencilla. Fue adquirida por Snowflake hace pocos años debido a su potencial y por eso disponéis de una sección dedicada a apps en la consola de Snowflake.

![streamlit](img/sis-example-app.png)

Referencia: https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit

Esto no quita que podamos generar una app de forma local y totalmente gratuita. Necesitaremos crear un proyecto con la librería `streamlit` https://docs.streamlit.io/get-started/installation

Una vez instalado podemos validar su funcionamiento mediante

```sh
streamlit hello
```

_**NOTA**: Streamlit es una librería de Python pero también funciona a nivel de comando si estamos en un terminal con el entorno de python activado. Si usáis UV será simplemente poner `uv run` por delante para que emplee las dependencias del entorno._

Podéis inicialmente crear un simple script (llamémosle _app.py_) e importar la librería `streamlit`.

```py
import streamlit as st

st.write("""

# Welcome!

This is my first app.

""")
```

Lanzando la aplicación podremos crear un servidor web que nos transformará el código markdown en código HTML para que nuestro navegador pueda visualizarlo. Podemos ver ejemplos de los comandos disponibles en la documentación de streamlit: https://docs.streamlit.io/get-started/installation

Quizás una de las formas más sencillas de familiarizarse es mediante la galería de Apps: https://streamlit.io/gallery

Existen multitud de componentes con los que deberemos familiarizarnos: https://docs.streamlit.io/get-started/fundamentals/main-concepts#widgets

## Gradio

Quizás algo más sencillo que Streamlit, Gradio se creo con el foco puesto en modelos de Machine Learning y fue muy respaldado por comunidades como HuggingFace donde se emplea para la sección de aplicaciones (Spaces) que esta alberga.

![grad](img/gradio.png)

A diferencia de Streamlit, Gradio si que permite ser embebido en un notebook de forma sencilla. Y veremos que también expone una API por defecto, así no solo los humanos si no que los programas pueden comunicarse con nuestra aplicación.

In [2]:
# !pip install gradio

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
)

demo.launch(share=True) # Share true comparte nuestro servicio local expuesto a internet

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://0238e1744fe76d9bf1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Pero como veis se trata de un servicio desplegado que emplea el puerto 7860 para publicar nuestra aplicación. Siempre dispone de un botón Submit ya que entiende que se trata de una aplicación que recibe algo de información y publica una respuesta.

En este mismo puerto se expone una API destinada al consumo programático del mismo recurso.

In [3]:
from gradio_client import Client

client = Client("http://127.0.0.1:7860/")
result = client.predict(
		name="iraitz",
		intensity=3,
		api_name="/predict"
)
print(result)

Loaded as API: http://127.0.0.1:7860/ ✔
Hello, iraitz!!!


De modo que es muy sencillo contar con una aplicación que nos de las piezas básicas necesarias a la hora de exponer un modelo capaz de predecir o clasificar una muestra. Tenéis una guía sencilla de uso en la siguiente ubicación:

https://www.gradio.app/guides/quickstart 

No os olvidéis de parar el servidor para que no consuma recursos o se quede como un proceso zombie en vuestra máquina.

In [4]:
demo.close()

Closing server running on port: 7861


Este ejemplo es una forma sencilla de exponer un modelo, que en este caso entrenaremos al vuelo pero podría haber sido entrenado de antemano.

In [6]:
import gradio as gr
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Train a decision tree classifier
model = DecisionTreeClassifier()
model.fit(X, y)

# Define the core prediction function
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    features = [[sepal_length, sepal_width, petal_length, petal_width]]
    prediction = model.predict(features)
    class_name = iris.target_names[prediction]
    return class_name

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_iris,
    inputs=["number", "number", "number", "number"],
    outputs="text",
    live=True,
)
iface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [7]:
iface.close()

Closing server running on port: 7861
